# How To: Write your own printer to use within SModelS

In [1]:
# Set up the path to SModelS installation folder
import sys; sys.path.append("."); import importlib; importlib.import_module("smodels_paths") if importlib.util.find_spec("smodels_paths") else None


<module 'smodels_paths' from '/home/lessa/smodels/docs/manual/source/recipes/smodels_paths.py'>

### The Printer

In [2]:
from smodels.tools.printers.basicPrinter import BasicPrinter
from smodels.matching.theoryPrediction import TheoryPredictionList
import os

class ExamplePrinter(BasicPrinter):
    """ A simple example of a custom printer, which only prints the analysis ID and the r-value """
    
    def __init__ ( self, output : str = "file", 
                   filename : str = "my.file" ):
        """ constructor, do what you will.
        In this example, we cachine in a self.toPrint
        object cache, write when .flush is called
        """
        BasicPrinter.__init__ ( self, output, filename )
        self.toPrint = []

    def setOutPutFile( self, filename : os.PathLike, overwrite : bool = True,       
        silent : bool = False ): 
        """ need to implement. Can implement your own logic here 
        :param filename: slha filename
        :param overwrite: does the user want to overwrite?
        :param silent: usually used to comment on removing old files
        """

        pass
    
    def addObj ( self, obj ):
        """ add an object, either do something immediately with it,
        or write to an object cache 
        """
        self.toPrint.append ( obj )
        
    def _formatTheoryPredictionList(self, obj: object) -> dict:
        """
        Format data of the TheoryPredictionList object.

        :param obj: A TheoryPredictionList object to be printed.
        """
        obj.sortTheoryPredictions()
        
        outputDict = {}
        for theoryPrediction in obj._theoryPredictions:
            expID = theoryPrediction.analysisId()
            r = theoryPrediction.getRValue()
            outputDict[expID] = r
            
        return outputDict
            

    def flush ( self ) -> dict:
        """ this method is called at the end of a model point """
        if hasattr(self,"rmin") and self.rmin > 0:
            rmin = self.rmin
        else:
            rmin = 0.0

        printerOutput = f"Example printer (r > {rmin:1.1e}):\n"
        for obj in self.toPrint:
            if not isinstance(obj,TheoryPredictionList):
                continue
            output = self._formatObj(obj)
            if not output:
                continue
            if not isinstance(output,dict):
                continue
            for expID,r in output.items():
                if r < rmin:
                    continue
                printerOutput += f"{expID} : r-value = {r:1.3f}\n"
        print(printerOutput)
        return {}

# these lines register the printer with smodels, to handle the "example" extension
# The handle name has to be defined in [printer]:outputType in order for the printer to be called.
from smodels.tools.printers.printerRegistry import PrinterRegistry                  
PrinterRegistry.register ( ExamplePrinter, "example" )



True

### Set up modelTester

In [3]:
from smodels.matching import modelTester
from smodels.experiment.databaseObj import Database

### Set up SModelS

In [4]:
# Set the path to the database
database = Database("official")
# database.getExpResults ()

In [5]:
parameterFile = "parameters_custom_printer.ini"
parser = modelTester.getParameters( parameterFile )

### Configure printers

In [6]:
## parameters_custom_printer.ini contains a section to configure your new printer:
# [example-printer]
# someArg = "this argument was set in parameters_custom_printer.ini"

In [7]:
## make sure to list your new printer in the list of printers used:
# [printer]
# outputType = example  ; use the example printer

In [8]:
modelTester.loadDatabaseResults(parser, database) 

In [9]:
fileList, inDir = modelTester.getAllInputFiles( "inputFiles/slha/simplyGluino.slha" )

In [10]:
### Run SModelS, it will call ExamplePrinter

In [11]:
modelTester.testPoints ( fileList, inDir, "results/", parser, database, timeout=0, development=False, parameterFile = parameterFile ) 

Example printer (r > 2.0e+01):
CMS-SUS-16-033 : r-value = 34.120
CMS-SUS-16-036 : r-value = 154.893
CMS-SUS-19-006 : r-value = 121.758
CMS-SUS-19-006-agg : r-value = 43.235



### Note how the printer can also be used via runSModelS.py directly, by adding to your ini file:

In [12]:
## custom code to be executed                                        
# [custom-codes]                                                              
# files = ./examplePrinter.py  # comma separated